# MTGFlow-CNF  —  Multivariate Time-Series Anomaly Detection
**Replaces the discrete MAF in MTGFlow with a Continuous Normalizing Flow (CNF)**

Pipeline:
```
Raw time-series
      ↓
Step 1 → Install dependencies
      ↓
Step 2 → Environment / imports / seed
      ↓
Step 3 → Load YAML configs + resolve dataset paths
      ↓
Step 4 → Cache windows per entity to disk  (skip if already cached)
      ↓
Step 5 → Setup MTGFlow-CNF model + helpers
      ↓
Step 6 → Train / Evaluate  (one entity or all)
```

In [ ]:
# ── STEP 1 ── Install dependencies ────────────────────────────────────────
import sys
!{sys.executable} -m pip install torchdiffeq --quiet   # required for CNF ODE solver
!{sys.executable} -m pip install matplotlib  --quiet
print("✅ Dependencies ready")

In [ ]:
# ── STEP 2 ── Environment / imports / seed ────────────────────────────────
import os, sys, gc, time, json, math, random, warnings, yaml
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score, precision_score, recall_score,
)

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── Paths  (edit these two lines if your layout differs) ──────────────────
MTAD_TOOLS_PATH  = "/home/ec2-user/mtad-gat-independent"
MTGFLOW_REPO_PATH = "/home/ec2-user/mtad-gat-independent/MTGFLOW-main"

assert os.path.isdir(MTAD_TOOLS_PATH),   f"Not found: {MTAD_TOOLS_PATH}"
assert os.path.isdir(MTGFLOW_REPO_PATH), f"Not found: {MTGFLOW_REPO_PATH}"

# Keep mtad-tools first so its 'utils' shadows any clash
sys.path = [p for p in sys.path if p not in [MTAD_TOOLS_PATH, MTGFLOW_REPO_PATH]]
sys.path.insert(0, MTAD_TOOLS_PATH)
sys.path.insert(1, MTGFLOW_REPO_PATH)

from dataloader import load_dataset
from data_preprocess import generate_windows

# ── Import MTGFlow-CNF (must live in MTGFLOW-main/models/MTGFLOW_CNF.py) ──
from models.MTGFLOW_CNF import build_mtgflow_cnf

print("✅ MTGFLOW_REPO_PATH :", os.path.abspath(MTGFLOW_REPO_PATH))
print("✅ MTGFlowCNF import  : OK")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Step 3 — Config selection + dataset resolution

In [ ]:
# ── STEP 3A ── Config files  (edit these) ────────────────────────────────
EXP_ID           = "mtad_gat_SMD"   # key in mtad_gat.yaml
DATASET_CFG_FILE = "SMD.yaml"       # your dataset YAML

MODEL_CFG_FILE   = "./mtad_gat.yaml"
assert os.path.exists(MODEL_CFG_FILE),   f"Missing: {MODEL_CFG_FILE}"
assert os.path.exists(DATASET_CFG_FILE), f"Missing: {DATASET_CFG_FILE}"

# ── CNF hyper-parameters  (tune freely) ───────────────────────────────────
CNF_CFG = dict(
    hidden_size    = 64,        # LSTM / GNN hidden dim
    ode_hidden_dim = 128,       # ODE MLP width  (hidden_size × 2 is a good start)
    ode_layers     = 3,         # ODE MLP depth
    dropout        = 0.0,       # LSTM dropout
    solver         = "dopri5",  # adaptive solver; use "euler" for fast debugging
    rtol           = 1e-4,
    atol           = 1e-5,
    kinetic_weight = 1e-3,      # set 0 to disable kinetic-energy regularisation
    adjoint        = False,     # True = memory-efficient backprop (slower)
)

# ── Training hyper-parameters ─────────────────────────────────────────────
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 256
EPOCHS       = 40
GRAD_CLIP    = 1.0      # gradient *norm* clip (more stable than value clip for CNF)

print("CNF config   :", CNF_CFG)
print("LR / WD / BS / Epochs :", LR, WEIGHT_DECAY, BATCH_SIZE, EPOCHS)

In [ ]:
# ── STEP 3B ── Load + normalise YAML configs ─────────────────────────────
with open(MODEL_CFG_FILE,   "r") as f: model_cfg   = yaml.safe_load(f)
with open(DATASET_CFG_FILE, "r") as f: dataset_cfg = yaml.safe_load(f)

# model_cfg may be flat or have a nested 'model' key
model_params = model_cfg.get("model", model_cfg) if isinstance(model_cfg, dict) else model_cfg

# Pull Base + dataset-specific section
assert "Base" in dataset_cfg, f"Expected 'Base' key in {DATASET_CFG_FILE}"
base = dataset_cfg["Base"]

# Resolve dataset name
DATASET_NAME = str(base.get("dataset", "unknown"))

# Merge Base with any dataset-override section (e.g. "SMD": {valid_ratio: 0})
_override_key = [k for k in dataset_cfg if k not in ("Base",)]
_override = dataset_cfg.get(_override_key[0], {}) if _override_key else {}
valid_ratio = float(_override.get("valid_ratio", 0.0))

# Resolve data root  (repo layout: ./data/<DATASET>/)
data_root = "./data/" + base.get("dataset", "SMD").upper() + "/"
if not os.path.isdir(data_root):
    data_root = str(base.get("data_root", data_root))
assert os.path.isdir(data_root), f"data_root not found: {data_root}"

# Window params from model YAML
window_size = int(model_params.get("window_size", 100))
stride      = int(model_params.get("stride", 1))

# Entity list  (take all by default; trim with [:N] for quick tests)
all_entities = list(base["entities"])

# Cache directory
CACHE_DIR = os.path.join("./cache_windows", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(CACHE_DIR, exist_ok=True)

# Results directory  (CNF results land here, separate from original MAF results)
RESULTS_ROOT = os.path.join("./results_mtgflow_cnf", DATASET_NAME, f"T{window_size}_S{stride}")
os.makedirs(RESULTS_ROOT, exist_ok=True)

print(f"Dataset      : {DATASET_NAME}")
print(f"data_root    : {os.path.abspath(data_root)}")
print(f"window_size  : {window_size}  stride: {stride}")
print(f"entities     : {len(all_entities)} total  first 5: {all_entities[:5]}")
print(f"CACHE_DIR    : {os.path.abspath(CACHE_DIR)}")
print(f"RESULTS_ROOT : {os.path.abspath(RESULTS_ROOT)}")

## Step 4 — Cache windows per entity to disk
Identical to the original notebook — skips entities that are already cached.

In [ ]:
# ── STEP 4 ── Build + cache windows  (idempotent) ────────────────────────
# Edit the slice below to cache more / fewer entities:
#   all_entities[:1]   → one entity only (fast test)
#   all_entities       → all entities
entities_to_cache = all_entities[:1]

print(f"Caching {len(entities_to_cache)} entit(y/ies) → {CACHE_DIR}")

for i, entity in enumerate(entities_to_cache):
    cache_path = os.path.join(CACHE_DIR, f"{entity}.npz")
    if os.path.exists(cache_path):
        print(f"[{i+1:02d}/{len(entities_to_cache)}] {entity}: already cached → skip")
        continue

    print(f"\n[{i+1:02d}/{len(entities_to_cache)}] Caching {entity} …")
    t0 = time.time()

    data_dict = load_dataset(
        data_root=data_root,
        entities=[entity],
        valid_ratio=valid_ratio,
        dim=int(base["dim"]),
        test_label_postfix=base["test_label_postfix"],
        test_postfix=base["test_postfix"],
        train_postfix=base["train_postfix"],
        nrows=base.get("nrows", None),
    )

    test_label_point = None
    try:
        test_label_point = data_dict[entity].get("test_label", None)
    except Exception:
        pass

    window_dict = generate_windows(data_dict, window_size=window_size, stride=stride)
    w = window_dict[entity]

    train_windows = w["train_windows"].astype(np.float32)
    test_windows  = w["test_windows"].astype(np.float32)

    if   "test_label"  in w: test_labels_pt = w["test_label"]
    elif "test_labels" in w: test_labels_pt = w["test_labels"]
    elif "label"       in w: test_labels_pt = w["label"]
    else: raise KeyError(f"No label key in {list(w.keys())}")

    test_labels_pt  = test_labels_pt.astype(np.float32)
    test_labels_win = (test_labels_pt.sum(axis=1) > 0).astype(np.int64)

    train_starts = np.asarray(w.get("train_starts",
        np.arange(train_windows.shape[0], dtype=np.int64) * stride), dtype=np.int64)
    test_starts  = np.asarray(w.get("test_starts",
        np.arange(test_windows.shape[0],  dtype=np.int64) * stride), dtype=np.int64)

    np.savez_compressed(
        cache_path,
        train_windows=train_windows,
        test_windows=test_windows,
        test_labels_window=test_labels_win,
        test_labels_point_window=test_labels_pt,
        train_starts=train_starts,
        test_starts=test_starts,
        test_label_point=(test_label_point if test_label_point is not None
                          else np.array([], dtype=np.int8)),
        meta=np.array([DATASET_NAME, entity, window_size, stride], dtype=object),
    )

    anom = float(test_labels_win.mean()) if test_labels_win.size else 0.0
    print(f"  train {train_windows.shape}  test {test_windows.shape}"
          f"  anomaly_ratio={anom:.3f}  elapsed={time.time()-t0:.1f}s")
    print(f"  saved → {cache_path}")

    del data_dict, window_dict, w, train_windows, test_windows, test_labels_pt
    gc.collect()

print("\n✅ Window caching complete.")

## Step 5 — Helper functions: dataset, model builder, train loop, evaluation, metrics

All shared utilities are defined in this single cell so any cell below can
call them independently after a kernel restart.

In [ ]:
# ── STEP 5 ── Helpers  ────────────────────────────────────────────────────
from torch.nn.utils import clip_grad_norm_

# ─────────────────────────────────────────────────────────────────────────────
# 5-A  Cache loader
# ─────────────────────────────────────────────────────────────────────────────
def load_cached_entity(entity: str) -> dict:
    cache_path = os.path.join(CACHE_DIR, f"{entity}.npz")
    assert os.path.exists(cache_path), f"Missing cache: {cache_path}  — run Step 4 first."
    d = np.load(cache_path, allow_pickle=True)
    return {
        "cache_path"              : cache_path,
        "train_windows"           : d["train_windows"],             # [N,T,K]
        "test_windows"            : d["test_windows"],              # [N,T,K]
        "test_labels_window"      : d["test_labels_window"],        # [N]
        "test_labels_point_window": d["test_labels_point_window"],  # [N,T]
        "train_starts"            : d["train_starts"],
        "test_starts"             : d["test_starts"],
        "test_label_point"        : (d["test_label_point"]
                                     if "test_label_point" in d.files
                                     else np.array([], dtype=np.int8)),
        "meta"                    : d["meta"] if "meta" in d.files else None,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 5-B  Dataset  (identical layout to original notebook)
# ─────────────────────────────────────────────────────────────────────────────
class MTGFlowWindowDataset(Dataset):
    """[N,T,K] cached windows → [K,T,1] tensors expected by MTGFlow-CNF."""
    def __init__(self, windows, labels=None, starts=None):
        self.windows = windows.astype(np.float32)
        self.labels  = None if labels is None else labels.astype(np.int64)
        self.starts  = None if starts is None else starts.astype(np.int64)

    def __len__(self): return self.windows.shape[0]

    def __getitem__(self, idx):
        x = torch.from_numpy(self.windows[idx]).transpose(0, 1).unsqueeze(-1)  # [K,T,1]
        y = 0  if self.labels is None else int(self.labels[idx])
        s = -1 if self.starts is None else int(self.starts[idx])
        return x, y, int(idx), s


def make_loaders(pack: dict):
    train_ds = MTGFlowWindowDataset(pack["train_windows"])
    test_ds  = MTGFlowWindowDataset(pack["test_windows"],
                                    labels=pack["test_labels_window"],
                                    starts=pack["test_starts"])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    return train_loader, test_loader


# ─────────────────────────────────────────────────────────────────────────────
# 5-C  Model builder
# ─────────────────────────────────────────────────────────────────────────────
def build_model(K: int, T: int) -> torch.nn.Module:
    """Build MTGFlowCNF from the global CNF_CFG dict."""
    model = build_mtgflow_cnf(K=K, T=T, device=str(DEVICE), **CNF_CFG)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  MTGFlowCNF  K={K}  T={T}  params={n:,}")
    return model


# ─────────────────────────────────────────────────────────────────────────────
# 5-D  Evaluation  (returns NLL anomaly scores — higher = more anomalous)
# ─────────────────────────────────────────────────────────────────────────────
def eval_window_scores(model, loader):
    """Returns (scores [N], labels [N], starts [N])."""
    model.eval()
    scores, labels, starts = [], [], []
    with torch.no_grad():
        for x, y, _, s in loader:
            x   = x.to(DEVICE)
            nll = -model.test(x)            # [N]  — NLL as anomaly score
            scores.append(nll.cpu().numpy())
            labels.append(np.asarray(y).reshape(-1))
            starts.append(np.asarray(s).reshape(-1))
    return (np.concatenate(scores),
            np.concatenate(labels),
            np.concatenate(starts))


# ─────────────────────────────────────────────────────────────────────────────
# 5-E  Metrics helpers  (identical to original notebook)
# ─────────────────────────────────────────────────────────────────────────────
def best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if thresholds.size == 0:
        return {"best_threshold": None, "best_f1": np.nan,
                "best_precision": np.nan, "best_recall": np.nan}
    p, r  = precision[:-1], recall[:-1]
    f1    = (2 * p * r) / (p + r + 1e-12)
    best  = int(np.nanargmax(f1))
    return {"best_threshold": float(thresholds[best]),
            "best_f1"       : float(f1[best]),
            "best_precision": float(p[best]),
            "best_recall"   : float(r[best])}


def windows_to_point_scores(window_scores, window_starts, series_len, window_size, agg="max"):
    if agg == "max":
        ps = np.full(series_len, -np.inf, dtype=np.float64)
    else:
        ps  = np.zeros(series_len, dtype=np.float64)
        cnt = np.zeros(series_len, dtype=np.int64)
    for s, sc in zip(window_starts, window_scores):
        s, e = int(s), min(series_len, int(s) + int(window_size))
        if e <= s: continue
        if agg == "max": ps[s:e] = np.maximum(ps[s:e], float(sc))
        else:
            ps[s:e]  += float(sc)
            cnt[s:e] += 1
    if agg == "max":
        ps[np.isneginf(ps)] = float(np.min(window_scores)) if window_scores.size else 0.0
    else:
        cov = cnt > 0
        ps[cov]  /= cnt[cov]
        ps[~cov]  = float(np.min(window_scores)) if window_scores.size else 0.0
    return ps


def windows_to_point_labels(test_labels_pt_win, test_starts, series_len, window_size):
    pl = np.zeros(series_len, dtype=np.int8)
    for i, s in enumerate(test_starts):
        s, e = int(s), min(series_len, int(s) + int(window_size))
        if e <= s: continue
        pl[s:e] = np.maximum(pl[s:e], (test_labels_pt_win[i][:(e-s)] > 0).astype(np.int8))
    return pl


def topk_windows(window_scores, window_starts, window_size, k=20):
    idx = np.argsort(-window_scores)[:k]
    return [{"rank": r+1, "window_index": int(i),
             "start": int(window_starts[i]),
             "end"  : int(window_starts[i]) + int(window_size),
             "score": float(window_scores[i])}
            for r, i in enumerate(idx)]


# ─────────────────────────────────────────────────────────────────────────────
# 5-F  Save results  (identical schema to original notebook — drop-in)
# ─────────────────────────────────────────────────────────────────────────────
def compute_and_save_results(entity, pack, window_scores, window_labels,
                             window_starts, out_dir, topk=20):
    os.makedirs(out_dir, exist_ok=True)
    y, s = window_labels.astype(int), window_scores.astype(float)

    auroc_w = roc_auc_score(y, s)             if y.size and np.unique(y).size > 1 else float("nan")
    aupr_w  = average_precision_score(y, s)   if y.size and np.unique(y).size > 1 else float("nan")
    bf1     = best_f1_threshold(y, s)
    thr     = bf1["best_threshold"]

    if thr is None:
        yhat = np.zeros_like(y)
        f1_w = p_w = r_w = float("nan")
    else:
        yhat = (s >= thr).astype(int)
        f1_w = float(f1_score(y, yhat, zero_division=0))
        p_w  = float(precision_score(y, yhat, zero_division=0))
        r_w  = float(recall_score(y, yhat, zero_division=0))

    # Point-level reconstruction
    tlp = pack.get("test_label_point", np.array([], dtype=np.int8))
    if tlp.size > 0:
        series_len  = int(tlp.shape[0])
        point_labels = (tlp > 0).astype(np.int8)
    else:
        series_len   = int(window_starts.max() + window_size) if window_starts.size else int(window_size)
        point_labels = windows_to_point_labels(
            pack["test_labels_point_window"], window_starts, series_len, window_size)

    ps_max  = windows_to_point_scores(s, window_starts, series_len, window_size, "max")
    ps_mean = windows_to_point_scores(s, window_starts, series_len, window_size, "mean")

    auroc_p = roc_auc_score(point_labels, ps_max)           if point_labels.size and np.unique(point_labels).size > 1 else float("nan")
    aupr_p  = average_precision_score(point_labels, ps_max) if point_labels.size and np.unique(point_labels).size > 1 else float("nan")

    metrics = {
        "dataset": DATASET_NAME, "entity": entity,
        "model"  : "MTGFlowCNF",
        "window_size": int(window_size), "stride": int(stride),
        "num_windows": int(s.size),
        "positive_windows": int(y.sum()) if y.size else 0,
        "anomaly_ratio_window": float(y.mean()) if y.size else 0.0,
        "auroc_window": float(auroc_w), "aupr_window": float(aupr_w),
        "best_threshold_f1_window": thr,
        "best_f1_window"      : float(bf1["best_f1"]),
        "best_precision_window": float(bf1["best_precision"]),
        "best_recall_window"  : float(bf1["best_recall"]),
        "f1_at_best_thr_window"        : float(f1_w),
        "precision_at_best_thr_window" : float(p_w),
        "recall_at_best_thr_window"    : float(r_w),
        "series_len"    : int(series_len),
        "auroc_point_max": float(auroc_p),
        "aupr_point_max" : float(aupr_p),
        "topk_windows"  : topk_windows(s, window_starts, window_size, k=topk),
    }

    with open(os.path.join(out_dir, f"{entity}_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    pd.DataFrame([metrics]).to_csv(os.path.join(out_dir, f"{entity}_metrics.csv"), index=False)
    np.savez_compressed(os.path.join(out_dir, f"{entity}_scores_windows.npz"),
        window_scores=s.astype(np.float32), window_labels=y.astype(np.int8),
        window_pred=yhat.astype(np.int8),   window_starts=window_starts.astype(np.int32))
    np.savez_compressed(os.path.join(out_dir, f"{entity}_scores_points.npz"),
        point_scores_max=ps_max.astype(np.float32),
        point_scores_mean=ps_mean.astype(np.float32),
        point_labels=point_labels.astype(np.int8))

    print(f"[{entity}] AUROC(w)={auroc_w:.4f}  AUPR(w)={aupr_w:.4f}  "
          f"BestF1={metrics['best_f1_window']:.4f}  thr={thr} | "
          f"AUROC(p,max)={auroc_p:.4f}  AUPR(p,max)={aupr_p:.4f}")
    print(f"  saved → {out_dir}")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# 5-G  Train one entity  (main training loop)
# ─────────────────────────────────────────────────────────────────────────────
def train_one_entity(entity: str) -> dict:
    """
    Full train + evaluate + save cycle for one entity.
    Returns the metrics dict (same schema as the original notebook).
    """
    pack = load_cached_entity(entity)
    out_dir = os.path.join(RESULTS_ROOT, entity)
    os.makedirs(os.path.join(out_dir, "models"),  exist_ok=True)
    os.makedirs(os.path.join(out_dir, "history"), exist_ok=True)

    train_loader, test_loader = make_loaders(pack)

    K = int(pack["train_windows"].shape[-1])   # sensors
    T = int(pack["train_windows"].shape[1])    # window length

    model = build_model(K, T)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    # Cosine annealing helps CNF training converge smoothly
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 0.01)

    history = {"epoch": [], "train_loss": [], "auroc_window": [], "aupr_window": []}
    best_auroc, best_state = -1.0, None

    for ep in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for x, _, _, _ in train_loader:
            x = x.to(DEVICE)
            opt.zero_grad()
            # model.forward() = mean log-likelihood − kinetic_reg  →  negate for minimisation
            loss = -model(x)
            loss.backward()
            # Clip gradient *norm* (better than value-clip for CNFs with variable step sizes)
            clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            losses.append(float(loss.item()))

        sched.step()
        train_loss = float(np.mean(losses)) if losses else float("nan")

        scores, labels_w, starts = eval_window_scores(model, test_loader)
        if labels_w.size and np.unique(labels_w).size > 1:
            auroc_w = float(roc_auc_score(labels_w, scores))
            aupr_w  = float(average_precision_score(labels_w, scores))
        else:
            auroc_w = aupr_w = float("nan")

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss)
        history["auroc_window"].append(auroc_w)
        history["aupr_window"].append(aupr_w)

        print(f"  epoch {ep:03d}/{EPOCHS} | "
              f"loss={train_loss:+.4f} | "
              f"AUROC(w)={auroc_w:.4f} | "
              f"AUPR(w)={aupr_w:.4f} | "
              f"lr={sched.get_last_lr()[0]:.2e}")

        if np.isfinite(auroc_w) and auroc_w > best_auroc:
            best_auroc = auroc_w
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)

    # Switch to fast Euler solver for inference scoring
    model.set_solver("euler")
    scores, labels_w, starts = eval_window_scores(model, test_loader)

    # Save checkpoint
    ckpt_path = os.path.join(out_dir, "models", f"{entity}_cnf.pt")
    torch.save({
        "state_dict" : model.state_dict(),
        "K": K, "T": T,
        "cnf_config" : CNF_CFG,
        "best_auroc" : best_auroc,
        "dataset"    : DATASET_NAME,
        "entity"     : entity,
        "window_size": window_size,
        "stride"     : stride,
    }, ckpt_path)

    # Save history
    hist_path = os.path.join(out_dir, "history", f"{entity}_cnf_history.csv")
    pd.DataFrame(history).to_csv(hist_path, index=False)

    # Compute + save full metrics
    metrics = compute_and_save_results(
        entity=entity, pack=pack,
        window_scores=scores, window_labels=labels_w, window_starts=starts,
        out_dir=out_dir, topk=20)
    metrics["best_auroc_during_training"] = float(best_auroc)
    metrics["ckpt_path"]    = ckpt_path
    metrics["history_path"] = hist_path

    del model, opt, sched
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return metrics

print("✅ Step 5 helpers loaded.")

## Step 6 — Train / Evaluate

- Set `RUN_ALL = False` to train a **single entity** (fast, good for debugging).
- Set `RUN_ALL = True`  to train **all entities** sequentially.
- Change `ENTITY_TO_RUN` to pick a different machine/sensor group.

In [ ]:
# ── STEP 6 ── Train + Evaluate ────────────────────────────────────────────
RUN_ALL      = False                # True  →  all entities
ENTITY_TO_RUN = all_entities[0]    # used only when RUN_ALL = False

entities = list(all_entities) if RUN_ALL else [ENTITY_TO_RUN]
print(f"Entities to run ({len(entities)}):", entities)

summaries = []
for i, ent in enumerate(entities, start=1):
    print(f"\n{'='*60}")
    print(f"  [{i}/{len(entities)}]  Training entity: {ent}")
    print(f"{'='*60}")
    m = train_one_entity(ent)
    summaries.append(m)

summary_df = pd.DataFrame(summaries)
overall_csv = os.path.join(RESULTS_ROOT, "_ALL_ENTITIES_SUMMARY.csv")
summary_df.to_csv(overall_csv, index=False)
print(f"\n✅ Saved overall summary → {overall_csv}")
summary_df

## Step 7 (optional) — Per-sensor anomaly scores

MTGFlowCNF exposes a `locate()` method that returns a score per sensor per window.
This can be used for fault localisation without any extra computation.

In [ ]:
# ── STEP 7 (optional) ── Per-sensor localisation ──────────────────────────
# Reload best checkpoint and score test windows at sensor granularity.

ENTITY_TO_LOCATE = all_entities[0]
pack = load_cached_entity(ENTITY_TO_LOCATE)
K = int(pack["test_windows"].shape[-1])
T = int(pack["test_windows"].shape[1])

ckpt_path = os.path.join(RESULTS_ROOT, ENTITY_TO_LOCATE,
                         "models", f"{ENTITY_TO_LOCATE}_cnf.pt")
assert os.path.exists(ckpt_path), f"Run Step 6 first — checkpoint not found: {ckpt_path}"

ckpt  = torch.load(ckpt_path, map_location=DEVICE)
model = build_model(K, T)
model.load_state_dict(ckpt["state_dict"])
model.set_solver("euler")   # fast for inference
model.eval()

test_ds     = MTGFlowWindowDataset(pack["test_windows"],
                                   labels=pack["test_labels_window"],
                                   starts=pack["test_starts"])
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

all_sensor_scores, all_labels, all_starts = [], [], []
with torch.no_grad():
    for x, y, _, s in test_loader:
        sensor_scores, _ = model.locate(x.to(DEVICE))  # [N, K]
        all_sensor_scores.append(sensor_scores.cpu().numpy())
        all_labels.append(np.asarray(y).reshape(-1))
        all_starts.append(np.asarray(s).reshape(-1))

sensor_scores_np = np.concatenate(all_sensor_scores)  # [N_windows, K]
labels_np        = np.concatenate(all_labels)

# Window-level score = mean over sensors  (matches overall anomaly score)
window_scores_from_sensors = sensor_scores_np.mean(axis=1)

# Show top-5 most anomalous sensors (averaged over anomalous windows)
anom_mask = labels_np == 1
if anom_mask.sum() > 0:
    mean_sensor_score_in_anomalies = sensor_scores_np[anom_mask].mean(axis=0)
    top5_sensor_idx = np.argsort(-mean_sensor_score_in_anomalies)[:5]
    print(f"Top-5 most anomalous sensors (by mean NLL in labelled anomaly windows):")
    for rank, idx in enumerate(top5_sensor_idx, 1):
        print(f"  [{rank}] sensor {idx:03d}  score={mean_sensor_score_in_anomalies[idx]:.4f}")
else:
    print("No anomaly-labelled windows in test set for this entity.")

# Heatmap: sensor × window  (first 500 windows for readability)
N_plot = min(500, sensor_scores_np.shape[0])
fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)
im = axes[0].imshow(sensor_scores_np[:N_plot].T, aspect="auto", cmap="hot_r",
                    origin="lower")
axes[0].set_ylabel("Sensor index")
axes[0].set_title(f"Per-sensor NLL anomaly score  ({ENTITY_TO_LOCATE})")
plt.colorbar(im, ax=axes[0])

axes[1].plot(labels_np[:N_plot], color="red", lw=1, label="Ground truth label")
axes[1].set_ylabel("Anomaly label")
axes[1].set_xlabel("Window index")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

del model
gc.collect()

## Step 8 (optional) — Plot training curve

In [ ]:
# ── STEP 8 (optional) ── Training curve ───────────────────────────────────
ENTITY_TO_PLOT = all_entities[0]
hist_path = os.path.join(RESULTS_ROOT, ENTITY_TO_PLOT,
                         "history", f"{ENTITY_TO_PLOT}_cnf_history.csv")
assert os.path.exists(hist_path), f"Run Step 6 first — history not found: {hist_path}"

hist = pd.read_csv(hist_path)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist["epoch"], hist["train_loss"], label="Train NLL loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title(f"MTGFlowCNF — Training Loss  ({ENTITY_TO_PLOT})")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist["epoch"], hist["auroc_window"], label="AUROC (window)", color="tab:blue")
axes[1].plot(hist["epoch"], hist["aupr_window"],  label="AUPR  (window)", color="tab:orange")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_title(f"MTGFlowCNF — Eval Metrics  ({ENTITY_TO_PLOT})")
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()